# Build i2Nav-Robot Corridor Dataset v1

Генерация датасета формата `image -> image-space trajectory/corridor mask` для i2Nav-Robot.

Структура выхода сделана совместимой с Wayomo-style corridor dataset:

```text
prepared_i2nav_robot_corridor_dataset_v1/
  images/
  corridor_mask/
  centerline_mask/
  gaussian_heatmap/
  overlays/
  meta/
    manifest.jsonl
    dataset_config.json
    sequences_summary.json
    generation_log.txt
    progress_live.json
  splits/
    train.txt
    val.txt
    test.txt
```

Основной режим: `EVERY_N = 20`, примерно 6067 валидных примеров по dry-run.


In [7]:
# =========================
# FINAL EXPORT CONFIG: i2Nav-Robot corridor dataset v1
# =========================

from pathlib import Path
import os
import sys
import json
import math
import time
import traceback
import subprocess
import importlib.util
from datetime import datetime
from collections import Counter

import numpy as np
import yaml
from PIL import Image, ImageDraw, ImageFilter
from IPython.display import display

PROJECT_ROOT = Path("/home/Jupyter/datasets/tesla/Waymo_open_dataset")
I2NAV_ROOT = PROJECT_ROOT / "external_datasets" / "i2Nav-Robot" / "sample_sequence" / "i2Nav-Robot"
OUT_ROOT = PROJECT_ROOT / "prepared_i2nav_robot_corridor_dataset_v1_2_sec"

DATASET_NAME = "i2nav_robot_corridor_leftcam_v1"
LEFT_TOPIC = "/avt_camera/left/image/compressed"

EVERY_N = 10
HORIZON_SEC = 2.0
STEP_SEC = 0.25

RUN_FULL_EXPORT = True
SAVE_OVERLAYS = True
CLEAN_OUTPUT = True

MIN_VALID_POINTS = 4
MIN_VALID_RATIO = 0.10
MIN_LONGEST_VALID_RUN = 4
REQUIRE_FIRST_POINT_VALID = False

MASK_LINE_WIDTH = 28
CENTERLINE_WIDTH = 8
OVERLAY_LINE_WIDTH = 10
POINT_RADIUS = 5
MAX_PIXEL_JUMP = 450

EXPECTED_CANDIDATES = 6906
EXPECTED_VALID = 6067

TRAIN_SEQS = [
    "building00",
    "building01",
    "building02",
    "parking00",
    "playground00",
    "street00",
    "street01",
]
VAL_SEQS = ["parking01"]
TEST_SEQS = ["parking02", "street02"]
ALL_SEQS = TRAIN_SEQS + VAL_SEQS + TEST_SEQS

print("PROJECT_ROOT:", PROJECT_ROOT, PROJECT_ROOT.exists())
print("I2NAV_ROOT:", I2NAV_ROOT, I2NAV_ROOT.exists())
print("OUT_ROOT:", OUT_ROOT)
print("ALL_SEQS:", ALL_SEQS)
assert I2NAV_ROOT.exists(), f"i2Nav-Robot dataset not found: {I2NAV_ROOT}"


PROJECT_ROOT: /home/Jupyter/datasets/tesla/Waymo_open_dataset True
I2NAV_ROOT: /home/Jupyter/datasets/tesla/Waymo_open_dataset/external_datasets/i2Nav-Robot/sample_sequence/i2Nav-Robot True
OUT_ROOT: /home/Jupyter/datasets/tesla/Waymo_open_dataset/prepared_i2nav_robot_corridor_dataset_v1_2_sec
ALL_SEQS: ['building00', 'building01', 'building02', 'parking00', 'playground00', 'street00', 'street01', 'parking01', 'parking02', 'street02']


In [8]:
# =========================
# Dependencies
# =========================

def pip_install(pkg):
    print(f"[pip] installing {pkg}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if importlib.util.find_spec("rosbags") is None:
    pip_install("rosbags")

if importlib.util.find_spec("cv2") is None:
    pip_install("opencv-python-headless")

import cv2
from rosbags.highlevel import AnyReader

print("deps ok")


deps ok


In [9]:
# =========================
# Calibration and geometry
# =========================

CALIB_PATH = I2NAV_ROOT / "calibration.yaml"
assert CALIB_PATH.exists(), CALIB_PATH

with open(CALIB_PATH, "r", encoding="utf-8") as f:
    calib = yaml.safe_load(f)

left_calib = calib["camera"]["left"]
fx, fy, cx, cy = left_calib["intrinsic"]
dist = np.array(left_calib["distortions"], dtype=np.float64)

K = np.array([
    [fx, 0.0, cx],
    [0.0, fy, cy],
    [0.0, 0.0, 1.0],
], dtype=np.float64)

# В calibration.yaml: Pi = R_i_c * Pc + t_i_c.
# T_imu_cam переводит camera -> IMU. Для проекции нужен обратный transform: IMU -> camera.
T_imu_cam = np.array(left_calib["T_imu_cam"], dtype=np.float64)
T_cam_imu = np.linalg.inv(T_imu_cam)

# Центр робота на полу в IMU frame.
odo_lever_imu = np.array(calib["adis16465"]["odo_lever"], dtype=np.float64)
CAM_RESOLUTION = tuple(calib["camera"]["resolution"])

print("CALIB_PATH:", CALIB_PATH)
print("K:\n", K)
print("dist:", dist)
print("T_imu_cam:\n", T_imu_cam)
print("T_cam_imu:\n", T_cam_imu)
print("odo_lever_imu:", odo_lever_imu)
print("CAM_RESOLUTION:", CAM_RESOLUTION)


def quat_xyzw_to_R(q):
    x, y, z, w = q
    n = math.sqrt(x * x + y * y + z * z + w * w)
    x, y, z, w = x / n, y / n, z / n, w / n

    xx, yy, zz = x * x, y * y, z * z
    xy, xz, yz = x * y, x * z, y * z
    wx, wy, wz = w * x, w * y, w * z

    return np.array([
        [1 - 2 * (yy + zz), 2 * (xy - wz), 2 * (xz + wy)],
        [2 * (xy + wz), 1 - 2 * (xx + zz), 2 * (yz - wx)],
        [2 * (xz - wy), 2 * (yz + wx), 1 - 2 * (xx + yy)],
    ], dtype=np.float64)


def make_T_world_imu(pos, quat_xyzw):
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = quat_xyzw_to_R(quat_xyzw)
    T[:3, 3] = pos
    return T


def invert_T(T):
    R = T[:3, :3]
    t = T[:3, 3]
    out = np.eye(4, dtype=np.float64)
    out[:3, :3] = R.T
    out[:3, 3] = -R.T @ t
    return out


def nearest_idx(times, t):
    return int(np.argmin(np.abs(times - t)))


def project_points_cam(points_cam):
    points_cam = np.asarray(points_cam, dtype=np.float64).reshape(-1, 1, 3)
    rvec = np.zeros((3, 1), dtype=np.float64)
    tvec = np.zeros((3, 1), dtype=np.float64)
    uv, _ = cv2.projectPoints(points_cam, rvec, tvec, K, dist)
    return uv.reshape(-1, 2)


def ros_stamp_to_float(stamp):
    if hasattr(stamp, "sec"):
        sec = stamp.sec
    elif hasattr(stamp, "secs"):
        sec = stamp.secs
    else:
        sec = 0

    if hasattr(stamp, "nanosec"):
        nsec = stamp.nanosec
    elif hasattr(stamp, "nsecs"):
        nsec = stamp.nsecs
    else:
        nsec = 0

    return float(sec) + float(nsec) * 1e-9


def load_traj(traj_path):
    # В i2Nav файл называется .csv, но фактически разделитель whitespace.
    try:
        arr = np.loadtxt(traj_path, delimiter=",")
    except Exception:
        arr = np.loadtxt(traj_path)

    assert arr.ndim == 2 and arr.shape[1] >= 8, f"Bad trajectory shape: {arr.shape}"
    return {
        "raw": arr,
        "t": arr[:, 0],
        "pos": arr[:, 1:4],
        "quat": arr[:, 4:8],  # xyzw
    }


def get_traj_idx_from_relative_time(traj_t, frame_rows_first_time, image_time):
    # ROS image timestamp и trajectory timestamp в разных абсолютных шкалах.
    # Поэтому синхронизируем относительно начала sequence.
    image_rel = float(image_time - frame_rows_first_time)
    traj_rel = traj_t - traj_t[0]
    idx = nearest_idx(traj_rel, image_rel)
    return idx, image_rel


def future_floor_points_world(traj, start_idx, horizon_sec, step_sec):
    traj_t = traj["t"]
    traj_pos = traj["pos"]
    traj_quat = traj["quat"]

    t0 = traj_t[start_idx]
    target_times = np.arange(t0 + step_sec, t0 + horizon_sec + 1e-9, step_sec)

    pts_world = []
    used_indices = []

    for tt in target_times:
        j = nearest_idx(traj_t, tt)
        T_w_i = make_T_world_imu(traj_pos[j], traj_quat[j])
        p_imu = np.array([odo_lever_imu[0], odo_lever_imu[1], odo_lever_imu[2], 1.0], dtype=np.float64)
        p_w = T_w_i @ p_imu
        pts_world.append(p_w[:3])
        used_indices.append(j)

    return np.asarray(pts_world, dtype=np.float64), used_indices


def project_future_trajectory_to_image(traj, image, image_time, first_image_time, horizon_sec, step_sec):
    w, h = image.size
    start_idx, image_rel = get_traj_idx_from_relative_time(
        traj_t=traj["t"],
        frame_rows_first_time=first_image_time,
        image_time=image_time,
    )

    T_w_i0 = make_T_world_imu(traj["pos"][start_idx], traj["quat"][start_idx])
    T_i0_w = invert_T(T_w_i0)

    pts_w, used_indices = future_floor_points_world(
        traj=traj,
        start_idx=start_idx,
        horizon_sec=horizon_sec,
        step_sec=step_sec,
    )

    pts_cam = []
    for p_w in pts_w:
        p_w_h = np.array([p_w[0], p_w[1], p_w[2], 1.0], dtype=np.float64)
        p_i0 = T_i0_w @ p_w_h          # world -> current IMU
        p_c = T_cam_imu @ p_i0         # current IMU -> current left camera
        pts_cam.append(p_c[:3])

    pts_cam = np.asarray(pts_cam, dtype=np.float64)
    uv = project_points_cam(pts_cam)

    valid = (
        (pts_cam[:, 2] > 0.1)
        & (uv[:, 0] >= 0)
        & (uv[:, 0] < w)
        & (uv[:, 1] >= 0)
        & (uv[:, 1] < h)
    )

    return {
        "uv": uv,
        "valid": valid,
        "pts_cam": pts_cam,
        "traj_idx": int(start_idx),
        "image_rel": float(image_rel),
        "used_indices": used_indices,
    }


CALIB_PATH: /home/Jupyter/datasets/tesla/Waymo_open_dataset/external_datasets/i2Nav-Robot/sample_sequence/i2Nav-Robot/calibration.yaml
K:
 [[1.0648950e+03 0.0000000e+00 8.0140490e+02]
 [0.0000000e+00 1.0652546e+03 6.2468780e+02]
 [0.0000000e+00 0.0000000e+00 1.0000000e+00]]
dist: [-1.516e-01  9.420e-02  1.690e-04 -1.420e-04 -2.290e-02]
T_imu_cam:
 [[ 0.00549192 -0.00206914  0.99998278  0.09      ]
 [ 0.99998051 -0.00295932 -0.00549803 -0.2       ]
 [ 0.00297065  0.99999348  0.00205284 -0.054     ]
 [ 0.          0.          0.          1.        ]]
T_cam_imu:
 [[ 0.00549192  0.9999805   0.00297065  0.19966224]
 [-0.00206913 -0.00295932  0.99999348  0.053594  ]
 [ 0.99998278 -0.00549803  0.00205285 -0.0909872 ]
 [ 0.          0.          0.          1.        ]]
odo_lever_imu: [0.    0.    0.852]
CAM_RESOLUTION: (1600, 1200)


In [10]:
# =========================
# Drawing, filtering, and progress helpers
# =========================

def max_valid_run(valid):
    best = 0
    cur = 0
    for v in valid:
        if bool(v):
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return best


def valid_segments_from_uv(uv, valid, max_pixel_jump=450):
    """
    Делит projected trajectory на непрерывные видимые сегменты.
    Не соединяет точки через valid=False и через большие pixel jumps.
    """
    segments = []
    current = []

    for i in range(len(uv)):
        if not bool(valid[i]):
            if len(current) >= 2:
                segments.append(current)
            current = []
            continue

        pt = (float(uv[i, 0]), float(uv[i, 1]))

        if current:
            prev = current[-1]
            jump = ((pt[0] - prev[0]) ** 2 + (pt[1] - prev[1]) ** 2) ** 0.5
            if jump > max_pixel_jump:
                if len(current) >= 2:
                    segments.append(current)
                current = [pt]
            else:
                current.append(pt)
        else:
            current = [pt]

    if len(current) >= 2:
        segments.append(current)

    return segments


def sample_is_good(proj):
    valid = proj["valid"]
    valid_count = int(valid.sum())
    total_count = int(len(valid))
    valid_ratio = valid_count / max(total_count, 1)
    longest_run = max_valid_run(valid)

    if longest_run < MIN_LONGEST_VALID_RUN:
        return False, f"short_valid_run:{longest_run}"

    if valid_count < MIN_VALID_POINTS:
        return False, f"too_few_valid_points:{valid_count}/{total_count}"

    if valid_ratio < MIN_VALID_RATIO:
        return False, f"low_valid_ratio:{valid_ratio:.3f}"

    if REQUIRE_FIRST_POINT_VALID and not bool(valid[0]):
        return False, "first_point_not_valid"

    return True, "ok"


def draw_binary_line_mask(size, uv, valid, line_width, max_pixel_jump=450):
    mask = Image.new("L", size, 0)
    draw = ImageDraw.Draw(mask)

    segments = valid_segments_from_uv(uv, valid, max_pixel_jump=max_pixel_jump)

    for pts in segments:
        if len(pts) < 2:
            continue
        draw.line(pts, fill=255, width=line_width, joint="curve")
        r = max(2, line_width // 2)
        for x, y in pts:
            draw.ellipse((x - r, y - r, x + r, y + r), fill=255)

    return mask


def draw_mask(size, uv, valid, line_width):
    return draw_binary_line_mask(size=size, uv=uv, valid=valid, line_width=line_width, max_pixel_jump=MAX_PIXEL_JUMP)


def draw_centerline_mask(size, uv, valid, line_width=CENTERLINE_WIDTH):
    return draw_binary_line_mask(size=size, uv=uv, valid=valid, line_width=line_width, max_pixel_jump=MAX_PIXEL_JUMP)


def draw_gaussian_heatmap(size, uv, valid, centerline_width=CENTERLINE_WIDTH, blur_radius=10):
    center = draw_centerline_mask(size=size, uv=uv, valid=valid, line_width=centerline_width)
    heat = center.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    return heat


def draw_overlay(image, uv, valid, meta_text, line_width, point_radius):
    overlay = image.copy()
    draw = ImageDraw.Draw(overlay)

    segments = valid_segments_from_uv(uv, valid, max_pixel_jump=MAX_PIXEL_JUMP)

    for pts in segments:
        if len(pts) < 2:
            continue
        draw.line(pts, fill=(255, 0, 0), width=line_width, joint="curve")
        for x, y in pts:
            r = point_radius
            draw.ellipse((x - r, y - r, x + r, y + r), fill=(0, 255, 0), outline=(0, 0, 0))

    box_w = min(1220, overlay.size[0] - 20)
    draw.rectangle((10, 10, box_w, 145), fill=(255, 255, 255), outline=(0, 0, 0), width=2)

    y = 20
    for line in meta_text[:6]:
        draw.text((20, y), line, fill=(0, 0, 0))
        y += 21

    return overlay


def ensure_final_dirs(out_root):
    dirs = {
        "images": out_root / "images",
        "corridor_mask": out_root / "corridor_mask",
        "centerline_mask": out_root / "centerline_mask",
        "gaussian_heatmap": out_root / "gaussian_heatmap",
        "overlays": out_root / "overlays",
        "meta": out_root / "meta",
        "splits": out_root / "splits",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def write_progress(progress_path, payload):
    tmp_path = progress_path.with_suffix(".json.tmp")
    tmp_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp_path.replace(progress_path)

print("helpers ready")


helpers ready


In [11]:
# =========================
# Final dataset generator with live logging
# =========================

def generate_i2nav_robot_corridor_dataset_final(
    out_root,
    sequences,
    every_n=20,
    horizon_sec=5.0,
    step_sec=0.25,
    save_overlays=True,
    clean_output=True,
    progress_update_every=25,
):
    out_root = Path(out_root)

    if clean_output and out_root.exists():
        import shutil
        print("[clean] removing:", out_root)
        shutil.rmtree(out_root)

    dirs = ensure_final_dirs(out_root)

    manifest_path = dirs["meta"] / "manifest.jsonl"
    log_path = dirs["meta"] / "generation_log.txt"
    summary_path = dirs["meta"] / "sequences_summary.json"
    config_path = dirs["meta"] / "dataset_config.json"
    progress_path = dirs["meta"] / "progress_live.json"

    manifest_rows = []
    sequence_summaries = []

    started_at = datetime.now().isoformat(timespec="seconds")
    t_global = time.time()

    split_by_seq = {}
    for seq in TRAIN_SEQS:
        split_by_seq[seq] = "train"
    for seq in VAL_SEQS:
        split_by_seq[seq] = "val"
    for seq in TEST_SEQS:
        split_by_seq[seq] = "test"

    config = {
        "dataset_name": DATASET_NAME,
        "source_dataset": "i2Nav-Robot",
        "source_root": str(I2NAV_ROOT),
        "output_root": str(out_root),
        "camera": "left",
        "camera_topic": LEFT_TOPIC,
        "input": "RGB image from left camera",
        "output": {
            "corridor_mask": "binary image-space trajectory/corridor mask",
            "centerline_mask": "thin binary centerline mask",
            "gaussian_heatmap": "blurred centerline heatmap",
        },
        "structure": "Wayomo-style compatible images + masks + manifest + splits",
        "image_resolution": "original camera resolution, no resize",
        "mask_resolution": "same as image",
        "every_n": every_n,
        "horizon_sec": horizon_sec,
        "step_sec": step_sec,
        "mask_line_width": MASK_LINE_WIDTH,
        "centerline_width": CENTERLINE_WIDTH,
        "overlay_line_width": OVERLAY_LINE_WIDTH,
        "max_pixel_jump": MAX_PIXEL_JUMP,
        "min_valid_points": MIN_VALID_POINTS,
        "min_valid_ratio": MIN_VALID_RATIO,
        "min_longest_valid_run": MIN_LONGEST_VALID_RUN,
        "require_first_point_valid": REQUIRE_FIRST_POINT_VALID,
        "expected_candidates_from_dry_run": EXPECTED_CANDIDATES,
        "expected_valid_from_dry_run": EXPECTED_VALID,
        "train_sequences": TRAIN_SEQS,
        "val_sequences": VAL_SEQS,
        "test_sequences": TEST_SEQS,
        "started_at": started_at,
    }
    config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

    def log(msg):
        line = f"[{datetime.now().isoformat(timespec='seconds')}] {msg}"
        print(line)
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(line + "\n")

    totals = {
        "seen_frames": 0,
        "candidate_frames": 0,
        "saved_samples": 0,
        "skipped_samples": 0,
        "errors": 0,
        "skip_reasons": Counter(),
    }

    def progress_payload(current_seq=None):
        elapsed = time.time() - t_global
        saved = totals["saved_samples"]
        speed = saved / elapsed if elapsed > 0 else 0.0
        remaining = max(EXPECTED_VALID - saved, 0)
        eta_sec = remaining / speed if speed > 0 else None

        return {
            "dataset_name": DATASET_NAME,
            "out_root": str(out_root),
            "current_seq": current_seq,
            "started_at": started_at,
            "updated_at": datetime.now().isoformat(timespec="seconds"),
            "elapsed_sec": round(elapsed, 2),
            "eta_sec": None if eta_sec is None else round(eta_sec, 2),
            "expected_valid": EXPECTED_VALID,
            "expected_candidates": EXPECTED_CANDIDATES,
            "totals": {
                "seen_frames": totals["seen_frames"],
                "candidate_frames": totals["candidate_frames"],
                "saved_samples": totals["saved_samples"],
                "skipped_samples": totals["skipped_samples"],
                "errors": totals["errors"],
                "skip_reasons": dict(totals["skip_reasons"]),
            },
            "sequences": sequence_summaries,
            "files": {
                "manifest": str(manifest_path),
                "log": str(log_path),
                "summary": str(summary_path),
                "config": str(config_path),
            },
        }

    log("=" * 120)
    log(f"START FINAL EXPORT: {DATASET_NAME}")
    log(f"out_root={out_root}")
    log(f"sequences={sequences}")
    log(f"every_n={every_n}, horizon_sec={horizon_sec}, step_sec={step_sec}")
    log(f"expected_valid≈{EXPECTED_VALID}")

    write_progress(progress_path, progress_payload(current_seq=None))

    for seq in sequences:
        seq_dir = I2NAV_ROOT / seq
        bag_path = seq_dir / f"{seq}.bag"
        traj_path = seq_dir / f"{seq}_trajectory.csv"
        gt_nav_path = seq_dir / f"{seq}_groundtruth.nav"

        seq_summary = {
            "seq": seq,
            "split": split_by_seq.get(seq, "unknown"),
            "bag_path": str(bag_path),
            "traj_path": str(traj_path),
            "gt_nav_path": str(gt_nav_path),
            "bag_exists": bag_path.exists(),
            "traj_exists": traj_path.exists(),
            "gt_nav_exists": gt_nav_path.exists(),
            "seen_frames": 0,
            "candidate_frames": 0,
            "saved_samples": 0,
            "skipped_samples": 0,
            "errors": 0,
            "skip_reasons": {},
            "started_at": datetime.now().isoformat(timespec="seconds"),
            "finished_at": None,
            "elapsed_sec": None,
        }

        sequence_summaries.append(seq_summary)
        seq_t0 = time.time()

        log("-" * 120)
        log(f"[{seq}] START split={seq_summary['split']}")
        log(f"[{seq}] bag={bag_path}")
        log(f"[{seq}] traj={traj_path}")

        if not bag_path.exists() or not traj_path.exists():
            reason = "missing_bag_or_traj"
            seq_summary["errors"] += 1
            totals["errors"] += 1
            seq_summary["skip_reasons"][reason] = seq_summary["skip_reasons"].get(reason, 0) + 1
            totals["skip_reasons"][reason] += 1
            log(f"[{seq}] SKIP: {reason}")
            continue

        try:
            traj = load_traj(traj_path)
            log(f"[{seq}] trajectory shape={traj['raw'].shape}, duration={traj['t'][-1] - traj['t'][0]:.2f}s")

            first_image_time = None
            local_saved = 0

            with AnyReader([bag_path]) as reader:
                conns = [c for c in reader.connections if c.topic == LEFT_TOPIC]
                if not conns:
                    reason = "left_topic_not_found"
                    seq_summary["errors"] += 1
                    totals["errors"] += 1
                    seq_summary["skip_reasons"][reason] = seq_summary["skip_reasons"].get(reason, 0) + 1
                    totals["skip_reasons"][reason] += 1
                    log(f"[{seq}] SKIP: {reason}")
                    continue

                msg_index = 0

                for conn, timestamp, rawdata in reader.messages(connections=conns):
                    seq_summary["seen_frames"] += 1
                    totals["seen_frames"] += 1

                    if msg_index % every_n != 0:
                        msg_index += 1
                        continue

                    seq_summary["candidate_frames"] += 1
                    totals["candidate_frames"] += 1

                    try:
                        msg = reader.deserialize(rawdata, conn.msgtype)
                        image_time = ros_stamp_to_float(msg.header.stamp) if hasattr(msg, "header") else float(timestamp) * 1e-9

                        if first_image_time is None:
                            first_image_time = image_time

                        arr = np.frombuffer(msg.data, dtype=np.uint8)
                        bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
                        if bgr is None:
                            reason = "decode_failed"
                            seq_summary["skipped_samples"] += 1
                            totals["skipped_samples"] += 1
                            seq_summary["skip_reasons"][reason] = seq_summary["skip_reasons"].get(reason, 0) + 1
                            totals["skip_reasons"][reason] += 1
                            msg_index += 1
                            continue

                        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                        image = Image.fromarray(rgb).convert("RGB")

                        proj = project_future_trajectory_to_image(
                            traj=traj,
                            image=image,
                            image_time=image_time,
                            first_image_time=first_image_time,
                            horizon_sec=horizon_sec,
                            step_sec=step_sec,
                        )

                        ok, reason = sample_is_good(proj)
                        if not ok:
                            seq_summary["skipped_samples"] += 1
                            totals["skipped_samples"] += 1
                            seq_summary["skip_reasons"][reason] = seq_summary["skip_reasons"].get(reason, 0) + 1
                            totals["skip_reasons"][reason] += 1
                            msg_index += 1
                            continue

                        sample_id = f"{seq}_{local_saved:06d}_msg_{msg_index:08d}"

                        image_rel = Path("images") / f"{sample_id}.png"
                        corridor_rel = Path("corridor_mask") / f"{sample_id}_corridor_mask.png"
                        centerline_rel = Path("centerline_mask") / f"{sample_id}_centerline_mask.png"
                        heatmap_rel = Path("gaussian_heatmap") / f"{sample_id}_gaussian_heatmap.png"
                        overlay_rel = Path("overlays") / f"{sample_id}_overlay.png"

                        image_path = out_root / image_rel
                        corridor_path = out_root / corridor_rel
                        centerline_path = out_root / centerline_rel
                        heatmap_path = out_root / heatmap_rel
                        overlay_path = out_root / overlay_rel

                        corridor_mask = draw_mask(image.size, proj["uv"], proj["valid"], MASK_LINE_WIDTH)
                        centerline_mask = draw_centerline_mask(image.size, proj["uv"], proj["valid"], CENTERLINE_WIDTH)
                        heatmap = draw_gaussian_heatmap(image.size, proj["uv"], proj["valid"], CENTERLINE_WIDTH, blur_radius=10)

                        image.save(image_path)
                        corridor_mask.save(corridor_path)
                        centerline_mask.save(centerline_path)
                        heatmap.save(heatmap_path)

                        valid_count = int(proj["valid"].sum())
                        total_points = int(len(proj["valid"]))
                        longest_run = int(max_valid_run(proj["valid"]))
                        segments = valid_segments_from_uv(proj["uv"], proj["valid"], max_pixel_jump=MAX_PIXEL_JUMP)

                        if save_overlays:
                            meta_text = [
                                f"{DATASET_NAME} seq={seq} split={seq_summary['split']}",
                                f"id={sample_id}",
                                f"msg_index={msg_index} image_rel={proj['image_rel']:.3f}s traj_idx={proj['traj_idx']}",
                                f"valid={valid_count}/{total_points} longest_run={longest_run} segments={len(segments)}",
                                f"horizon={horizon_sec}s step={step_sec}s",
                                f"z_cam={proj['pts_cam'][:, 2].min():.2f}..{proj['pts_cam'][:, 2].max():.2f}",
                            ]
                            overlay = draw_overlay(image, proj["uv"], proj["valid"], meta_text, OVERLAY_LINE_WIDTH, POINT_RADIUS)
                            overlay.save(overlay_path)

                        manifest_row = {
                            "id": sample_id,
                            "dataset_name": DATASET_NAME,
                            "source_dataset": "i2Nav-Robot",
                            "sequence": seq,
                            "split": seq_summary["split"],
                            "camera": "left",
                            "camera_topic": LEFT_TOPIC,
                            "input_type": "rgb_image",
                            "output_type": "image_space_future_trajectory_corridor",
                            "image_path": str(image_rel),
                            "corridor_mask_path": str(corridor_rel),
                            "centerline_mask_path": str(centerline_rel),
                            "gaussian_heatmap_path": str(heatmap_rel),
                            "overlay_path": str(overlay_rel) if save_overlays else None,
                            "source_bag_path": str(bag_path),
                            "source_traj_path": str(traj_path),
                            "source_gt_nav_path": str(gt_nav_path),
                            "bag_msg_index": int(msg_index),
                            "image_time": float(image_time),
                            "image_rel_time": float(proj["image_rel"]),
                            "traj_idx": int(proj["traj_idx"]),
                            "future_used_indices": [int(x) for x in proj["used_indices"]],
                            "horizon_sec": float(horizon_sec),
                            "step_sec": float(step_sec),
                            "num_future_points": int(total_points),
                            "valid_points": int(valid_count),
                            "valid_ratio": float(valid_count / max(total_points, 1)),
                            "longest_valid_run": int(longest_run),
                            "segments_count": int(len(segments)),
                            "max_pixel_jump": int(MAX_PIXEL_JUMP),
                            "image_width": int(image.size[0]),
                            "image_height": int(image.size[1]),
                            "mask_line_width": int(MASK_LINE_WIDTH),
                            "centerline_width": int(CENTERLINE_WIDTH),
                            "calibration_path": str(CALIB_PATH),
                        }

                        manifest_rows.append(manifest_row)
                        with open(manifest_path, "a", encoding="utf-8") as f:
                            f.write(json.dumps(manifest_row, ensure_ascii=False) + "\n")

                        seq_summary["saved_samples"] += 1
                        totals["saved_samples"] += 1
                        local_saved += 1

                        if totals["saved_samples"] % progress_update_every == 0:
                            write_progress(progress_path, progress_payload(current_seq=seq))
                            log(f"[progress] saved={totals['saved_samples']} candidates={totals['candidate_frames']} seen={totals['seen_frames']} current_seq={seq}")

                    except Exception as e:
                        reason = f"error:{type(e).__name__}"
                        seq_summary["errors"] += 1
                        totals["errors"] += 1
                        seq_summary["skip_reasons"][reason] = seq_summary["skip_reasons"].get(reason, 0) + 1
                        totals["skip_reasons"][reason] += 1
                        log(f"[{seq}] ERROR msg_index={msg_index}: {repr(e)}")
                        log(traceback.format_exc()[-2000:])

                    msg_index += 1

        except Exception as e:
            reason = f"fatal:{type(e).__name__}"
            seq_summary["errors"] += 1
            totals["errors"] += 1
            seq_summary["skip_reasons"][reason] = seq_summary["skip_reasons"].get(reason, 0) + 1
            totals["skip_reasons"][reason] += 1
            log(f"[{seq}] FATAL: {repr(e)}")
            log(traceback.format_exc())

        seq_summary["finished_at"] = datetime.now().isoformat(timespec="seconds")
        seq_summary["elapsed_sec"] = round(time.time() - seq_t0, 2)

        log(f"[{seq}] DONE: seen={seq_summary['seen_frames']} candidates={seq_summary['candidate_frames']} saved={seq_summary['saved_samples']} skipped={seq_summary['skipped_samples']} errors={seq_summary['errors']} elapsed={seq_summary['elapsed_sec']}s")
        log(f"[{seq}] skip_reasons={seq_summary['skip_reasons']}")
        write_progress(progress_path, progress_payload(current_seq=seq))

    split_rows = {"train": [], "val": [], "test": []}
    for row in manifest_rows:
        split_rows[row["split"]].append(row["id"])

    for split_name, ids in split_rows.items():
        split_path = dirs["splits"] / f"{split_name}.txt"
        split_path.write_text("\n".join(ids) + "\n", encoding="utf-8")

    finished_at = datetime.now().isoformat(timespec="seconds")

    summary = {
        "dataset_name": DATASET_NAME,
        "source_dataset": "i2Nav-Robot",
        "out_root": str(out_root),
        "started_at": started_at,
        "finished_at": finished_at,
        "elapsed_sec": round(time.time() - t_global, 2),
        "total_seen_frames": totals["seen_frames"],
        "total_candidate_frames": totals["candidate_frames"],
        "total_saved_samples": totals["saved_samples"],
        "total_skipped_samples": totals["skipped_samples"],
        "total_errors": totals["errors"],
        "skip_reasons": dict(totals["skip_reasons"]),
        "split_counts": {k: len(v) for k, v in split_rows.items()},
        "sequences": sequence_summaries,
        "paths": {
            "images": str(dirs["images"]),
            "corridor_mask": str(dirs["corridor_mask"]),
            "centerline_mask": str(dirs["centerline_mask"]),
            "gaussian_heatmap": str(dirs["gaussian_heatmap"]),
            "overlays": str(dirs["overlays"]),
            "manifest": str(manifest_path),
            "splits": str(dirs["splits"]),
            "log": str(log_path),
            "progress": str(progress_path),
            "config": str(config_path),
        },
    }

    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    write_progress(progress_path, progress_payload(current_seq=None))

    log("=" * 120)
    log(f"FINISHED: saved={totals['saved_samples']}, candidates={totals['candidate_frames']}, seen={totals['seen_frames']}, skipped={totals['skipped_samples']}, errors={totals['errors']}")
    log(f"manifest={manifest_path}")
    log(f"summary={summary_path}")
    log(f"progress={progress_path}")

    return {
        "manifest_rows": manifest_rows,
        "summary": summary,
        "manifest_path": manifest_path,
        "summary_path": summary_path,
        "log_path": log_path,
        "progress_path": progress_path,
        "config_path": config_path,
    }

print("generator ready")


generator ready


In [ ]:
# =========================
# Run full export: N=20
# =========================

if RUN_FULL_EXPORT:
    full_result = generate_i2nav_robot_corridor_dataset_final(
        out_root=OUT_ROOT,
        sequences=ALL_SEQS,
        every_n=EVERY_N,
        horizon_sec=HORIZON_SEC,
        step_sec=STEP_SEC,
        save_overlays=SAVE_OVERLAYS,
        clean_output=CLEAN_OUTPUT,
        progress_update_every=25,
    )

    print("FULL OUT_ROOT:", OUT_ROOT)
    print("saved samples:", len(full_result["manifest_rows"]))
    print("manifest:", full_result["manifest_path"])
    print("summary:", full_result["summary_path"])
    print("progress:", full_result["progress_path"])
    print("log:", full_result["log_path"])
else:
    print("RUN_FULL_EXPORT=False")


[2026-05-22T11:10:25] ========================================================================================================================
[2026-05-22T11:10:25] START FINAL EXPORT: i2nav_robot_corridor_leftcam_v1
[2026-05-22T11:10:25] out_root=/home/Jupyter/datasets/tesla/Waymo_open_dataset/prepared_i2nav_robot_corridor_dataset_v1_2_sec
[2026-05-22T11:10:25] sequences=['building00', 'building01', 'building02', 'parking00', 'playground00', 'street00', 'street01', 'parking01', 'parking02', 'street02']
[2026-05-22T11:10:25] every_n=20, horizon_sec=2.0, step_sec=0.25
[2026-05-22T11:10:25] expected_valid≈6067
[2026-05-22T11:10:25] ------------------------------------------------------------------------------------------------------------------------
[2026-05-22T11:10:25] [building00] START split=train
[2026-05-22T11:10:25] [building00] bag=/home/Jupyter/datasets/tesla/Waymo_open_dataset/external_datasets/i2Nav-Robot/sample_sequence/i2Nav-Robot/building00/building00.bag
[2026-05-22T11:10

KeyboardInterrupt: 

In [ ]:
# =========================
# Check live progress manually while generation is running
# =========================

progress_path = OUT_ROOT / "meta" / "progress_live.json"
print(progress_path)

if progress_path.exists():
    progress = json.loads(progress_path.read_text(encoding="utf-8"))
    print(json.dumps(progress["totals"], ensure_ascii=False, indent=2))
    print("current_seq:", progress["current_seq"])
    print("elapsed_sec:", progress["elapsed_sec"])
    print("eta_sec:", progress["eta_sec"])
else:
    print("progress file not found yet")


In [ ]:
# =========================
# Final sanity check after export
# =========================

summary_path = OUT_ROOT / "meta" / "sequences_summary.json"
manifest_path = OUT_ROOT / "meta" / "manifest.jsonl"

print("OUT_ROOT:", OUT_ROOT)
print("summary exists:", summary_path.exists())
print("manifest exists:", manifest_path.exists())

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "total_saved_samples": summary.get("total_saved_samples"),
        "total_candidate_frames": summary.get("total_candidate_frames"),
        "total_skipped_samples": summary.get("total_skipped_samples"),
        "total_errors": summary.get("total_errors"),
        "split_counts": summary.get("split_counts"),
        "skip_reasons": summary.get("skip_reasons"),
    }, ensure_ascii=False, indent=2))

if manifest_path.exists():
    n = sum(1 for _ in open(manifest_path, "r", encoding="utf-8"))
    print("manifest rows:", n)

for sub in ["images", "corridor_mask", "centerline_mask", "gaussian_heatmap", "overlays"]:
    d = OUT_ROOT / sub
    print(sub, len(list(d.glob("*.png"))) if d.exists() else "missing")


In [ ]:
# =========================
# Visual sample grid from generated overlays
# =========================

overlay_dir = OUT_ROOT / "overlays"
overlay_paths = sorted(overlay_dir.glob("*.png"))[:12]
print("overlay samples:", len(overlay_paths))

thumbs = []
for p in overlay_paths:
    img = Image.open(p).convert("RGB")
    img = img.resize((520, int(520 * img.size[1] / img.size[0])))
    thumbs.append(img)

if thumbs:
    cols = 2
    rows = math.ceil(len(thumbs) / cols)
    grid = Image.new("RGB", (cols * thumbs[0].width, rows * thumbs[0].height), (255, 255, 255))
    for i, img in enumerate(thumbs):
        grid.paste(img, ((i % cols) * img.width, (i // cols) * img.height))
    display(grid)
else:
    print("No overlays found")
